# DysonianLineCNN — Train and Evaluate

Thin wrapper around [`dyson_cnn`](../dyson_cnn/). All real logic lives in the Python package;
this notebook exists only to pin the training invocation and surface results in a Jupyter UI.

**Before running**, make sure `config/paths.json` points at the correct Drive-mounted project
folder (`DysonianLineCNN`) and that the synthetic dataset `X_dyson_mix_<Prefix>.npy` + friends
are present in that folder. See [`config/README.md`](../config/README.md) for details.

Profile is selected via `config/training.json` `active_profile` field, or the `DYSON_PROFILE`
environment variable (which takes precedence). On Mac, set `DYSON_PROFILE=mac_dev` for fast
debug iterations or `mac_full` for full-dataset local training.

## 1. Environment setup

Idempotent across Colab and local Mac. On Colab, mounts Google Drive. On Mac, verifies
that the notebook is run from the repository root.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure we are at the repo root so `dyson_cnn` imports resolve.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'dyson_cnn').is_dir():
    # Walk up until we find it (handles `notebooks/` as working directory).
    for parent in Path.cwd().parents:
        if (parent / 'dyson_cnn').is_dir():
            REPO_ROOT = parent
            os.chdir(REPO_ROOT)
            break
    else:
        raise RuntimeError(f'Could not find repo root from {Path.cwd()}')

IN_COLAB = 'google.colab' in sys.modules
print(f'Environment : {"Colab" if IN_COLAB else "local Mac"}')
print(f'Repo root    : {REPO_ROOT}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

## 2. Load all configs

Resolves `paths.json`, `dataset.json`, `training.json`, `inference.json`. Profile is taken
from `DYSON_PROFILE` if set, else `training.json.active_profile`.

In [ ]:
from dyson_cnn.config import load_all

paths, dataset_cfg, training_cfg, inference_cfg = load_all(
    REPO_ROOT / 'config',
    profile=os.environ.get('DYSON_PROFILE'),
)

print(f'Runtime         : {paths["runtime"]}')
print(f'Project dir     : {paths["project_dir"]}')
print(f'Profile         : {training_cfg["profile_name"]}')
print(f'Epochs / batch  : {training_cfg["epochs"]} / {training_cfg["batch_size"]}')
print(f'max_samples     : {training_cfg["max_samples"]}')
print(f'mixed_precision : {training_cfg["mixed_precision"]}')

## 3. Train

`train_and_save_run` is the single entry point. It loads the dataset, splits it, builds the
CNN, fits, and writes an atomic run directory containing:

- `cnn_model.keras`
- `y_min.npy` / `y_max.npy` (from train split only)
- `B_axis.npy` / `B_axis.csv`
- `model_meta.json` (generator metadata snapshot + training profile)
- `history.csv`
- `loss.png` / `loss_full_and_zoom.png`
- `_X_test.npy` / `_y_test.npy` (cached for step 4)

In [ ]:
from dyson_cnn.train import train_and_save_run

run_dir = train_and_save_run(paths, dataset_cfg, training_cfg)
print(f'\nSaved run: {run_dir}')

## 4. Evaluate on the test split

`evaluate_run` loads the saved model and the cached test split from the run directory and
produces per-head MAE/RMSE plus a parity plot.

In [ ]:
from dyson_cnn.evaluate import evaluate_run

result = evaluate_run(run_dir)
print(f'\nN test samples: {result["n_samples"]}')
for head, m in result['metrics'].items():
    print(f'  {head:>2}: MAE={m["mae"]:.6g}, RMSE={m["rmse"]:.6g}')

## 5. Next steps

1. Open [`config/inference.json`](../config/inference.json) and set `runName` to the newly
   created run's timestamp directory name.
2. Run [`02_infer_real.ipynb`](02_infer_real.ipynb) to predict Dysonian parameters from a
   real experimental spectrum CSV.
3. Open `matlab/Validator.m` to reconstruct the fitted curve and overlay it on the raw spectrum.